# Supporting Statistical Computation 
## Exercise on Malawi IHS-5

Complete each TODO cell. Add short written interpretation below each section.

## Setup

### 0.1 Imports and paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

In [ ]:
"""
PY4Africa - Beginner Course
Day 5: Statistical Computing with SciPy
Dataset: Malawi IHS-5 (2019-2020)
Files: hh_mod_a_filt.dta, HH_MOD_B.dta, HH_MOD_C.dta, ihs5_consumption_aggregate.dta
"""

# Try project root first, then notebook-relative path.
DATA_DIR = Path('../../data/_raw/IHS 5 DATA sample')

print('Input folder:', DATA_DIR)


### 0.2 Load data

In [ ]:
# Load source datasets with categorical labels enabled for readability.
hh = pd.read_stata(DATA_DIR / 'hh_mod_a_filt.dta', convert_categoricals=True)
roster = pd.read_stata(DATA_DIR / 'HH_MOD_B.dta', convert_categoricals=True)

# For this notebook we only need two education fields.
edu = pd.read_stata(
    DATA_DIR / 'HH_MOD_C.dta',
    convert_categoricals=True,
    columns=['case_id', 'PID', 'hh_c08', 'hh_c09']
)
cons = pd.read_stata(DATA_DIR / 'ihs5_consumption_aggregate.dta', convert_categoricals=True)

print('Loaded shapes:')
print('  hh    ', hh.shape)
print('  roster', roster.shape)
print('  edu   ', edu.shape)
print('  cons  ', cons.shape)


## Data preparation (shared across all exercises)

### 1) Household size and head profile

In [ ]:
# 1) Household size and head profile
# Household size = number of roster records per household.
hh_size = roster.groupby('case_id').size().rename('hh_size')

# Head is relationship code 1 (or label 'HEAD').
is_head = roster['hh_b04'].astype('string').str.strip().str.upper() == 'HEAD'

head = roster.loc[is_head, ['case_id', 'PID', 'hh_b05a', 'hh_b03']].copy()
head.columns = ['case_id', 'head_pid', 'head_age', 'head_sex']

# Keep these analysis fields clean.
head['head_age'] = pd.to_numeric(head['head_age'], errors='coerce')
# Keep raw head sex values; map only inside sections where needed.

### 2) Education and welfare fields

In [ ]:
# 2) Education + welfare fields
# Keep education as labeled categories in shared prep.
head_edu = edu[['case_id', 'PID', 'hh_c08', 'hh_c09']].copy()
head_edu.columns = ['case_id', 'head_pid', 'head_education_proxy', 'head_qualification_code']

# In this dataset, per-capita real consumption is rexpaggpc.
cons['pcrexpagg'] = cons['rexpaggpc'].copy()
cons_sel = cons[['case_id', 'rexpagg', 'pcrexpagg', 'poor']].copy()

# Keep raw poverty labels/codes; map in the section where needed.

### 3) Build final analysis table

In [ ]:
# 3) Assemble analysis frame
# Merge all household-level information into one table.
df = (
    hh
    .merge(hh_size, on='case_id', how='left')
    .merge(head, on='case_id', how='left')
    .merge(head_edu, on=['case_id', 'head_pid'], how='left')
    .merge(cons_sel, on='case_id', how='left')
)

# Keep urban/rural as labeled value from source.
df['urban_rural'] = df['reside']

# Convert core numeric analysis columns once.
for col in ['hh_size', 'head_age', 'rexpagg', 'pcrexpagg']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Households: {len(df):,}')
print('Columns from consumption aggregate (aliased): rexpagg, pcrexpagg, poor')
df.head()


## Quick visual diagnostics

In [ ]:
# Quick EDA plots before exercises
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1) Household size histogram
x1 = df['hh_size'].dropna()
axes[0, 0].hist(x1, bins=range(1, int(x1.max()) + 2), color='#4C78A8', edgecolor='white')
axes[0, 0].set_title('Household size distribution')
axes[0, 0].set_xlabel('hh_size')
axes[0, 0].set_ylabel('Households')

# 2) Household size boxplot
axes[0, 1].boxplot(x1, vert=True)
axes[0, 1].set_title('Household size boxplot')
axes[0, 1].set_ylabel('hh_size')

# 3) Per-capita consumption histogram (raw)
x2 = df['pcrexpagg'].dropna()
axes[1, 0].hist(x2.clip(upper=x2.quantile(0.99)), bins=50, color='#F58518', edgecolor='white')
axes[1, 0].set_title('Per-capita consumption (clipped at p99)')
axes[1, 0].set_xlabel('pcrexpagg')
axes[1, 0].set_ylabel('Households')

# 4) Per-capita consumption by urban/rural (boxplot)
grp_u = df.loc[df['urban_rural'] == 'URBAN', 'pcrexpagg'].dropna()
grp_r = df.loc[df['urban_rural'] == 'RURAL', 'pcrexpagg'].dropna()
axes[1, 1].boxplot([grp_u, grp_r], labels=['URBAN', 'RURAL'], showfliers=False)
axes[1, 1].set_title('Per-capita consumption by urban/rural')
axes[1, 1].set_ylabel('pcrexpagg')

plt.tight_layout()
plt.show()


## Exercise 1 — Exploring Distribution Objects

Use SciPy's normal distribution object in code and call its core methods.
Print outputs clearly and check if generated values make practical sense.

### 1.1 The four methods on a normal distribution

Create a fitted normal object (`fitted`) from `pcrexpagg` and call `rvs`, `pdf`, `cdf`, and `ppf`.
Store each output in variables and print them in a readable format.

In [ ]:
# TODO: fit a normal distribution to pcrexpagg
# 1) compute mean/std
# 2) create fitted = stats.norm(loc=..., scale=...)
# 3) call rvs, pdf, cdf, ppf and print results


### 1.2 Comparing CDF with actual data

Pick one threshold and compute both theoretical share (`fitted.cdf`) and actual share (`(x <= threshold).mean()`).
Print the gap and repeat quickly with one more threshold.

In [ ]:
# TODO: compare theoretical vs observed shares below a threshold
# 1) choose a threshold (e.g., median)
# 2) compute fitted.cdf(threshold)
# 3) compute (x <= threshold).mean() and compare


## Exercise 2 — Confidence Intervals for the Mean

Compute 95% CIs for mean per-capita consumption at national and subgroup levels.
Focus on writing reusable code and comparing interval width/overlap.

### 2.1 National CI for per-capita consumption

In [ ]:
# TODO: compute the national 95% CI for mean pcrexpagg
# 1) x = df['pcrexpagg'].dropna(), then n, mean_x, se
# 2) use stats.t.interval(0.95, df=n-1, loc=mean_x, scale=se)
# 3) print n, mean, CI, and MOE%


### 2.2 CIs by region

In [ ]:
# TODO: compute 95% CI by region
# 1) loop over each region
# 2) for each subset, compute n, mean, se, CI, and MOE%
# 3) print a compact comparison table


### 2.3 Urban vs rural CI comparison

In [ ]:
# TODO: compare 95% CIs for urban vs rural pcrexpagg
# 1) clean/map urban_rural labels in this cell
# 2) compute n, mean, se, and CI for each group
# 3) print both intervals and state whether they overlap


## Exercise 3 — Confidence Level and Sample Size

Show how CI width changes with confidence level and sample size.
Build compact result tables so patterns are easy to compare.

### 3.1 Changing the confidence level

In [ ]:
# TODO: 3.1 Changing the confidence level
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


### 3.2 Effect of sample size on precision

In [ ]:
# TODO: 3.2 Effect of sample size on precision
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


## Exercise 4 (stretch) — CI for a Proportion

Estimate poverty-rate proportions and their confidence intervals.
Repeat by urban/rural and compute the CI for the difference in proportions.

### 4.1 Poverty rate with confidence interval

In [ ]:
# TODO: 4.1 Poverty rate with confidence interval
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


### 4.2 Poverty rate by urban/rural + difference CI

In [ ]:
# TODO: 4.2 Poverty rate by urban/rural + difference CI
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


## Exercise 5 — Outlier Detection

Flag outliers in size/consumption, then compare summary stats before vs after filtering.
Use tables and simple plots to explain impact of outlier handling.

### 5.1 IQR fences on household size

In [ ]:
# TODO: 5.1 IQR fences on household size
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


### 5.1 Plot task

In [ ]:
# TODO: make side-by-side boxplots for hh_size (all vs after removing IQR outliers)


### 5.2 MAD-based robust z-scores on per-capita consumption

In [ ]:
# TODO: 5.2 MAD-based robust z-scores on per-capita consumption
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


### 5.2 Plot task

In [ ]:
# TODO: histogram of robust z-scores with vertical lines at -3.5 and +3.5


### 5.3 Before-and-after comparison table for pcrexpagg

In [ ]:
# TODO: build table with metrics (n, mean, median, std)
# columns: All data, After IQR filter, After MAD filter


### 5.3 Plot task

In [ ]:
# TODO: bar chart comparing mean pcrexpagg across the three versions


## Exercise 6 — Hypothesis Testing

Run group-comparison tests (t-test, Mann-Whitney, chi-square) on welfare and poverty outcomes.
Report both statistical significance and practical effect size.

### 6.1 Urban vs rural per-capita consumption (Welch t-test)

In [ ]:
# TODO: 6.1 Urban vs rural per-capita consumption (Welch t-test)
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


### 6.1 Plot task

In [ ]:
# TODO: plot mean pcrexpagg by urban/rural with error bars (std error)


### 6.2 Non-parametric check (Mann-Whitney)

In [ ]:
# TODO: stats.mannwhitneyu(urban, rural, alternative='two-sided')


### 6.3 Urban vs rural household size (second t-test)

In [ ]:
# TODO: 6.3 Urban vs rural household size (second t-test)
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


### 6.4 Chi-square: poverty status × urban/rural

In [ ]:
# TODO: build crosstab rows=urban_rural cols=poor
sub = df[['urban_rural', 'poor']].copy()
sub['urban_rural'] = sub['urban_rural'].astype('string').str.strip().str.upper()

# TODO: map poor locally (labels or codes) -> poor_num (0/1)
# poor_txt = ...
# sub['poor_num'] = ...

# TODO: chi2_contingency + row percentages using poor_num


### 6.4 Plot task

In [ ]:
# TODO: stacked bar chart of poverty composition (poor=0/1) by urban/rural


### 6.5 One-sample t-test benchmark (mean hh_size = 4.4)

In [ ]:
# TODO: 6.5 One-sample t-test benchmark (mean hh_size = 4.4)
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


## Exercise 7 — Correlation & Simple Regression

Measure associations and fit simple regression lines for welfare relationships.
Compare classical and robust slope estimates and interpret direction/strength.

### 7.1 Household size vs per-capita consumption

In [ ]:
# TODO: 7.1 Household size vs per-capita consumption
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


### 7.2 Age of head vs per-capita consumption

In [ ]:
# TODO: 7.2 Age of head vs per-capita consumption
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


### 7.3 Simple regression: pcrexpagg ~ hh_size

In [ ]:
# TODO: 7.3 Simple regression: pcrexpagg ~ hh_size
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.


### 7.3 Plot task

In [ ]:
# TODO: scatter plot (sample or alpha) and fitted OLS line


### 7.4 Robust regression comparison (Theil-Sen)

In [ ]:
# TODO: theilslopes and compare with OLS slope


### 7.5 Stretch: correlation matrix

In [ ]:
# TODO: 7.5 Stretch: correlation matrix
# 1) Prepare the analysis subset and handle missing values.
# 2) Run the required calculation/test for this subsection.
# 3) Print key results clearly and add your interpretation in markdown.
